# Pronóstico de demanda diaria

**Pregunta:** ¿cuántas ventas habrá en los próximos días y qué día conviene reforzar inventario o turnos?

> Por defecto este notebook usa **datos sintéticos** (`scripts/generate_sample_data.py`). No saques conclusiones de negocio hasta cambiar `DATA_PATH` a datos reales y anonimizados.

**Flujo:** carga y calidad → exploración → validación temporal → línea base → modelo → métricas → pronóstico → exportables.

## 0. Parámetros

In [ ]:
from pathlib import Path

DATA_PATH = Path("../data/ventas_ejemplo.csv")   # cambia por tus datos reales (columnas: fecha, ventas)
SEASON = 7        # estacionalidad semanal
TEST_DAYS = 56    # últimas 8 semanas como prueba
HORIZON = 14      # días a pronosticar hacia adelante

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from statsmodels.tsa.holtwinters import ExponentialSmoothing

pd.options.display.float_format = "{:,.2f}".format
DIAS = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]

## 1. Carga y calidad de los datos

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"No encuentro {DATA_PATH}. Genera datos de ejemplo con:  python scripts/generate_sample_data.py"
    )

df = pd.read_csv(DATA_PATH, parse_dates=["fecha"])
print(df.shape, df["fecha"].min().date(), "→", df["fecha"].max().date())
df.head()

In [ ]:
print("Nulos:")
print(df.isna().sum(), "\n")
print("Fechas duplicadas:", df["fecha"].duplicated().sum())

serie = (
    df.drop_duplicates("fecha")
      .set_index("fecha")["ventas"]
      .sort_index()
      .asfreq("D")            # fuerza frecuencia diaria; los huecos quedan como NaN
)
print("Días faltantes:", int(serie.isna().sum()))

serie = serie.interpolate(limit=3)   # solo rellena huecos cortos
assert serie.isna().sum() == 0, "Quedan huecos largos: revisa los datos antes de modelar"
serie.describe()

## 2. Exploración

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=serie.index, y=serie, name="Ventas diarias", line=dict(width=1, color="#9AA5B8")))
fig.add_trace(go.Scatter(x=serie.index, y=serie.rolling(7).mean(), name="Media móvil 7 días", line=dict(color="#0EA5E9", width=2)))
fig.update_layout(title="Ventas diarias", template="plotly_white", height=380, legend=dict(orientation="h"))
fig.show()

In [ ]:
por_dia = serie.groupby(serie.index.day_of_week).mean()
por_dia.index = DIAS

fig = go.Figure(go.Bar(x=por_dia.index, y=por_dia.values, marker_color="#185FA5"))
fig.update_layout(title="Venta promedio por día de la semana", template="plotly_white", height=320)
fig.show()
por_dia.round(1)

## 3. Validación temporal

Se entrena con el pasado y se prueba con el futuro (las últimas `TEST_DAYS`). Una división al azar filtraría información del futuro y daría resultados demasiado optimistas.

In [ ]:
train, test = serie.iloc[:-TEST_DAYS], serie.iloc[-TEST_DAYS:]
print(f"Entrenamiento: {train.index.min().date()} → {train.index.max().date()} ({len(train)} días)")
print(f"Prueba:        {test.index.min().date()} → {test.index.max().date()} ({len(test)} días)")

## 4. Línea base

"Mañana vendo lo mismo que el mismo día de la semana pasada". Un modelo solo vale la pena si **le gana** a esto.

In [ ]:
ultima_semana = train.iloc[-SEASON:].to_numpy()
base_pred = pd.Series(
    np.tile(ultima_semana, int(np.ceil(TEST_DAYS / SEASON)))[:TEST_DAYS],
    index=test.index,
    name="linea_base",
)
base_pred.head()

## 5. Modelo: Holt-Winters

Suavizado exponencial con tendencia amortiguada y estacionalidad semanal aditiva.

In [ ]:
modelo = ExponentialSmoothing(
    train, trend="add", damped_trend=True, seasonal="add", seasonal_periods=SEASON
).fit()

pred = pd.Series(modelo.forecast(TEST_DAYS).to_numpy(), index=test.index, name="holt_winters")
pred.head()

## 6. Métricas

- **MAE:** error absoluto medio, en las mismas unidades que las ventas.
- **WAPE:** error absoluto total entre ventas totales (en %). Es estable aunque haya días con ventas bajas.

In [ ]:
def metricas(real, pronostico):
    error = real - pronostico
    return {
        "MAE": error.abs().mean(),
        "WAPE %": 100 * error.abs().sum() / real.abs().sum(),
    }

tabla = pd.DataFrame({
    "Línea base": metricas(test, base_pred),
    "Holt-Winters": metricas(test, pred),
}).T

mejora = 1 - tabla.loc["Holt-Winters", "MAE"] / tabla.loc["Línea base", "MAE"]
display(tabla)
print(f"Mejora del MAE frente a la línea base: {mejora:.1%}")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=train.index[-120:], y=train.iloc[-120:], name="Histórico", line=dict(color="#9AA5B8")))
fig.add_trace(go.Scatter(x=test.index, y=test, name="Real (prueba)", line=dict(color="#0A0F1E")))
fig.add_trace(go.Scatter(x=base_pred.index, y=base_pred, name="Línea base", line=dict(dash="dot", color="#7B8499")))
fig.add_trace(go.Scatter(x=pred.index, y=pred, name="Holt-Winters", line=dict(color="#0EA5E9", width=3)))
fig.update_layout(title="Prueba: real vs. pronóstico", template="plotly_white", height=420, legend=dict(orientation="h"))
fig.show()

## 7. Pronóstico hacia adelante

Se reentrena con **toda** la historia y se pronostican los próximos `HORIZON` días.

In [ ]:
final = ExponentialSmoothing(
    serie, trend="add", damped_trend=True, seasonal="add", seasonal_periods=SEASON
).fit()

futuro = final.forecast(HORIZON).round(2).rename("pronostico")
futuro_df = futuro.to_frame()
futuro_df["dia"] = [DIAS[d] for d in futuro_df.index.day_of_week]
futuro_df

In [ ]:
resumen = futuro_df.groupby("dia")["pronostico"].mean().sort_values(ascending=False)
print(f"Demanda promedio esperada por día (próximos {HORIZON} días):")
print(resumen.round(1).to_string())
print(f"\nDía de mayor demanda esperada: {resumen.index[0]}  |  más flojo: {resumen.index[-1]}")

## 8. Exportables

Genera el gráfico interactivo y el CSV del pronóstico en `reports/` (el HTML se puede incrustar en el sitio con un `<iframe>`).

In [ ]:
REPORTS = Path("../reports")
REPORTS.mkdir(exist_ok=True)

fig_final = go.Figure()
fig_final.add_trace(go.Scatter(x=serie.index[-120:], y=serie.iloc[-120:], name="Histórico", line=dict(color="#9AA5B8")))
fig_final.add_trace(go.Scatter(x=futuro.index, y=futuro, name="Pronóstico", line=dict(color="#0EA5E9", width=3)))
fig_final.update_layout(
    title=f"Pronóstico de ventas: próximos {HORIZON} días",
    template="plotly_white", height=440, legend=dict(orientation="h"),
)
fig_final.write_html(REPORTS / "forecast.html", include_plotlyjs="cdn", full_html=True)
futuro_df.to_csv(REPORTS / "pronostico.csv", index_label="fecha")
print("Guardado en", REPORTS.resolve())

## 9. Conclusiones *(completar con datos reales)*

- **Resultado:** el modelo mejora el MAE en **[XX %]** frente a la línea base.
- **Decisión que habilita:** *[p. ej. reforzar inventario el día pico y reducir compra el día flojo]*.
- **Límites:** *[periodo corto, sin clima/promociones/festivos, datos sintéticos…]*.
- **Siguientes pasos:** *[agregar variables externas, probar otros modelos, medir impacto en merma]*.